# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to load, explore, and process a dataset defined by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset's metadata is described in Croissant format at the following URL:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Install `mlcroissant` if needed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and discover its structure using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL (FAIR^2 dataset)
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print out dataset name and description
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
Review the available record sets and their `@id` fields from the dataset schema using the Croissant API.

In [ ]:
# List all available record sets by @id and name
record_sets = list(dataset.record_sets())
if not record_sets:
    print("No record sets are defined in the Croissant metadata.")
else:
    print(f"Number of record sets found: {len(record_sets)}\n")
    for rs in record_sets:
        print(f"@id: {rs['@id']}\n  name: {rs['name']}\n  description: {rs.get('description', 'No description')}\n")

## 3. Data Extraction
Load records from each record set into Pandas DataFrames. All record set, field, and column references must use their Croissant `@id`.

In [ ]:
# Extract records for all discovered record sets
import pprint

dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

if not record_set_ids:
    print("No record sets present to extract data from.")
else:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for record set '{record_set_id}'")
            print(f"First few columns: {list(df.columns)[:5]}")
        else:
            print(f"No records found for record set: {record_set_id}")

    if dataframes:
        # Preview the first available dataframe
        first_rs_id = next(iter(dataframes.keys()))
        print(f"\nColumns in {first_rs_id}:")
        pprint.pprint(dataframes[first_rs_id].columns.tolist())
        print("\nData sample:")
        display(dataframes[first_rs_id].head(5))

## 4. Exploratory Data Analysis (EDA)
Explore and preprocess numeric fields. Example: filter, normalize, and group using field `@id` references. (You may adjust fields and thresholds as appropriate for your dataset.)

In [ ]:
# Example EDA: filter, normalize, and group by on the first available record set and numeric column
if not dataframes:
    print("No dataframes are loaded for EDA.")
else:
    # Select a dataframe to demonstrate (using the first one found)
    rs_id = next(iter(dataframes.keys()))
    df = dataframes[rs_id].copy()

    # Try to automatically select a numeric (float/int) column
    numeric_field_candidates = df.select_dtypes(include=['number']).columns.tolist()
    if not numeric_field_candidates:
        print("No numeric fields detected. Available columns are:")
        print(list(df.columns))
    else:
        # Use first numeric column (use @id as column name as per Croissant)
        numeric_field_id = numeric_field_candidates[0]
        print(f"Using numeric field (by @id): {numeric_field_id}\n")

        # Choose a threshold (example: mean value)
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)}):")
        display(filtered_df.head())

        # Normalize the numeric field
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, normalized_col]].head())

        # Try grouping by a categorical field (find first non-numeric @id column)
        cat_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        if cat_fields:
            group_field = cat_fields[0]
            print(f"\nGrouping by field (by @id): {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the numeric field and its normalized variant. Adjust column `@id` values as appropriate from your EDA section.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes or not numeric_field_candidates:
    print("No suitable data for visualization.")
else:
    # Visualize the original and normalized values
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True, ax=axs[0])
    axs[0].set_title(f"Distribution of {numeric_field_id}")
    axs[0].set_xlabel(numeric_field_id)

    sns.histplot(filtered_df[normalized_col].dropna(), bins=15, kde=True, ax=axs[1], color='orange')
    axs[1].set_title(f"Normalized {numeric_field_id} (filtered)")
    axs[1].set_xlabel(normalized_col)

    plt.tight_layout()
    plt.show()

    # Optionally, boxplot by group if grouped_df exists
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
- This notebook demonstrated how to load and process a dataset defined by a Croissant schema using `mlcroissant`.
- All references—including record sets and fields—used their Croissant `@id`s.
- We conducted initial data exploration, performed basic numeric EDA, and visualized variable distributions and categorical groupings.

**Next steps:** Repeat the EDA/visualization steps for other record sets or fields by referencing their `@id` values as seen above.